imports

In [24]:
import geocoder
import chromadb
import uuid
import time
import requests
import webbrowser
import sys
from sentence_transformers import SentenceTransformer

embedding model laden

In [3]:
embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 648.47it/s]


vector Database aanmaken

In [4]:
client = chromadb.PersistentClient(path="./recept_database")

collection = client.get_or_create_collection(
    name="recepten"
)

gekozen ingredienten invullen

In [6]:
items = []

print('')

while True:
    item = input("welke gerechten heb je al/ heb je die je wilt gebruiken. (type stop om te stoppen): ")

    if item == "stop":
        break

    items.append(item)

    

selecteren welke supermarkt je boodschappen doet

In [7]:
destination = input('wat is de supermarkt waar je heen wilt( vul supermarkt in of niks)')

In [8]:
destination = destination.lower()

if "alber" in destination or 'hein' in destination or destination == "ah":
    supermarkt = "https://www.ah.nl/zoeken?query="

elif "plus" in destination:
    supermarkt = "https://www.plus.nl/zoekresultaten?SearchTerm="

elif "jum" in destination:
    supermarkt = "https://www.jumbo.com/producten/?searchType=keyword&searchTerms="

elif "spar" in destination or 'spa' in destination:
    supermarkt = "https://www.spar.nl/zoek/?fq="

elif 'lid' in destination or 'lit' in destination or 'lil' in destination:
    supermarkt = 'https://www.lidl.nl/q/search?q='

else:
    supermarkt = "https://google.com/search?q=supermarkt+voor+"

???

In [9]:
g = geocoder.ip("me")
regio = g.city
lat = g.latlng[0]
lon = g.latlng[1]

Query voor de vector DB

In [10]:
def build_query(items):
    return f"""
Ik wil koken met: {', '.join(items)}.
Ik zoek gerechten die hierbij passen.
"""

Uitzoeken welke rcepten erop lijken

In [11]:
def get_similar_recipes(items, amount=5):

    query = build_query(items)
    embedding = embedding_model.encode(query)

    result = collection.query(
        query_embeddings=[embedding.tolist()],
        n_results=amount,
        where={
            "liked": True
        }
    )

    if len(result["documents"]) == 0:
        return result

    if len(result["documents"][0]) == 0:
        return result

    return result

Formatteren voor betere prompting

In [12]:
def format_similar(results):
    if not results["documents"]:
        return ""

    return "\n".join(results["documents"][0][:5])

Verglijkbare waardes in een var stoppen

In [13]:
print(lat)
print(lon)

results = get_similar_recipes(items)
similar_recipes = format_similar(results)

52.0908
5.1222


De standaard prompt

In [14]:
prompts = [
    f"""
Genereer EXACT 5 verschillende gerechten op basis van deze ingrediënten:

{items}

Vergelijkbare recepten uit eerdere sessies:

{similar_recipes}

BELANGRIJKE REGELS:
- Geef EXACT 5 regels terug.
- Geen uitleg voor of na de regels.
- Geen markdown.
- Geen opsommingstekens.
- Elke regel moet EXACT dit formaat hebben:

gerecht_nr | gerecht | ingredienten | duur | nieuw ingredient = (ingredient)

VOORBEELD:
1 | Pasta met kip | kip, pasta, ui | 25 min | nieuw ingredient = (parmezaanse kaas)

EXTRA REGELS:
- Gebruik zoveel mogelijk ingrediënten uit de opgegeven lijst.
- Maximaal 1 nieuw ingrediënt per gerecht.
- Bereidingstijd maximaal 30 minuten.
- Het nieuwe ingrediënt MOET tussen haakjes staan.
- De tekst 'nieuw ingredient = ...' moet altijd het laatste onderdeel van de regel zijn.
- Gebruik het pipe-teken '|' uitsluitend als scheidingsteken.
- Gebruik geen pipe-tekens in gerechtnamen of ingrediënten.
- Nummer de gerechten van 1 t/m 5.

Geef alleen de 5 regels terug.
"""
]

Recepten opslaan

In [15]:
def save_recipe(recipe_text):

    embedding = embedding_model.encode(recipe_text)

    collection.add(
    ids=[str(uuid.uuid4())],
    documents=[recipe_text],
    embeddings=[embedding.tolist()],
    metadatas=[{
        "ingredients" : ",".join(items),
        "liked": True
    }]
)

Tijd meten starten

In [17]:
for prompt in prompts:
    start_time = time.time()

MetaData opslaan

In [18]:
data = {
    "model": "nvidia/nemotron-3-nano-4b",
    "messages": [
        {"role": "user", "content": prompt}
    ],
    "temperature": 0.7
}


Bot functie

In [19]:
url = 'http://127.0.0.1:1234/v1/chat/completions'



def lmbot():
    try:
        response = requests.post(url, json=data)
        response.raise_for_status()

        end_time = time.time()

        result = response.json()

        answer = result["choices"][0]["message"]["content"]
        model_name = result.get("model", "onbekend")
        response_length = len(answer)

    except requests.exceptions.RequestException as e:
        print(f"Fout bij API-aanroep: {e}")

    regels = []

    for regel in answer.splitlines():
        if "nieuw ingredient" in regel.lower():
            ingredient = regel.split("=", 1)[1].strip()
            ingredient = ingredient.strip("()[]{}")
            ingredient = ingredient.replace(" ", "+")

            url2 = supermarkt + ingredient
            regel += f"|{url2}"

        regels.append(regel)

    return "\n".join(regels)

uitvoeren van de eerste stap van de agent

In [22]:
while True:
    bot = lmbot()

    print(bot)

    check = input("Zit hier een goed gerecht tussen? (Ja/Nee) ")

    if check.lower() == "ja":
        break


1 | Pasta met kip | kip, pasta | 25 min | nieuw ingredient = (parmezaanse kaas)  |https://www.jumbo.com/producten/?searchType=keyword&searchTerms=parmezaanse+kaas
2 | Kioek in rijst | kip, rijst | 30 min | nieuw ingredient = (parmezaanse kaas)  |https://www.jumbo.com/producten/?searchType=keyword&searchTerms=parmezaanse+kaas
3 | Kioeksoep | kip, eier | 25 min | nieuw ingredient = (parmezaanse kaas)  |https://www.jumbo.com/producten/?searchType=keyword&searchTerms=parmezaanse+kaas
4 | Geroosterde kioek met zup | kip, ui | 30 min | nieuw ingredient = (parmezaanse kaas)  |https://www.jumbo.com/producten/?searchType=keyword&searchTerms=parmezaanse+kaas
5 | Kioekstroop | kip, spinazie | 25 min | nieuw ingredient = (parmezaanse kaas)|https://www.jumbo.com/producten/?searchType=keyword&searchTerms=parmezaanse+kaas


Hele geselecteerde recept uitprinten

In [23]:
recept_nummer = int(input("Welk nummer van recept wil je hebben? "))

for regel in bot.splitlines():
    if regel.startswith(f"{recept_nummer} |"):
        recept_select = regel
        break
save_recipe(recept_select)

prompts = [
    f"""
    maak een recept voor:{recept_select}
    """
]

for prompt in prompts:
    start_time = time.time()

data = {
    "model": "nvidia/nemotron-3-nano-4b",
    "messages": [
        {"role": "user", "content": prompt}
    ],
    "temperature": 0.7
}

bot = lmbot()
print(bot)

full_recipe = bot
save_recipe(full_recipe)


**4 Geroosterde Kipborst met Zup & Parmezaanse Kaas**  

| Portie | Tijd | New‑ingredient |
|--------|------|----------------|
| 4 (≈ 600 g kip) | 30 min | 30 g parmaise kaas (fresh grated) |

### Ingrediënten
- 4 kipborsten (ca. 150 g elk)  
- 1 grote ui, in blokjes gesneden  
- 2 tl olijfolie  
- 1 tl parmezaan‑paprikapulver (of gewoon paprika)  
- ½ tl zout  
- ¼ tl witte peper, fijngedropt  
- **40 g parmaise kaas** (fresh grated) – het “nieuwe” ingrediënt!  

### Benodigde tools
- Oven  
- Kookpan  
- Spoon / roker voor de ui‑schor  
- Tussenvlaaier of wiskraam  

---

## Stapsgewijs recept (≈ 30 min)

1. **Vorkal de oven**  
   - Vervang 200 °C vooraf (voor een 15‑minutige warm‑up).  

2. **Marinade & roeren van de kip**  
   - Kik de paprikapulver, zout en peper over de kipborsten.  
   - Bestrijk ze met 1 tl olijfolie.  
   - Leg ze op een bakplaat (geen afvallen) en roosterd ze **20 min** tot de binnenkant witte en knapperig is, de buitenkant licht bruin.

3. **Ui‑schor maken

Supermarkt route

In [82]:
boy = input('Wil je een kaart naar de supermarkt in de buurt (ja/nee)')

if boy.lower() == 'ja':
    if destination in ("niks", ""):
        url2 = (
            f"https://www.google.com/maps/dir/?api=1"
            f"&origin={lat},{lon}"
            f"&destination=supermarkt"
            f"&travelmode=driving"
        )
        webbrowser.open(url2)

    else:
        url2 = (
            f"https://www.google.com/maps/dir/?api=1"
            f"&origin={lat},{lon}"
            f"&destination={destination}"
            f"&travelmode=driving"
        )

        webbrowser.open(url2)
else:
    sys.exit()

SystemExit: 

c:\AI-Agent\food_finder_ai_agent\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
